In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define file paths
data_path = Path("../data/lstm/processed/bitcoin_feature_engineered.csv")

models_dir = Path("../models")
model_path = models_dir / "lstm_model.keras"
feature_scaler_path = models_dir / "feature_scaler.pkl"
target_scaler_path = models_dir / "target_scaler.pkl"

prediction_dir = Path("../data/lstm/processed")
prediction_path = prediction_dir / "bitcoin_daily_predictions.csv"

# Define today's prediction date
today = pd.Timestamp.today().normalize()

print("Prediction date:", today.date())
print("Feature data path:", data_path)
print("Model path:", model_path)

Prediction date: 2026-10-29
Feature data path: ..\data\lstm\processed\bitcoin_feature_engineered.csv
Model path: ..\models\lstm_model.keras


In [3]:
# Load feature-engineered Bitcoin data
df = pd.read_csv(data_path)

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort data by date
df = df.sort_values("date").reset_index(drop=True)

# Exclude today's data and any future dates
df = df[df["date"] < today].copy()
df = df.reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Latest available historical date:", df["date"].max().date())
print("\nColumns:")
print(df.columns.tolist())

display(df.tail())

Dataset shape: (3204, 9)
Date range: 2018-01-20 to 2026-10-28
Latest available historical date: 2026-10-28

Columns:
['date', 'open', 'high', 'low', 'close', 'volume', 'RSI', 'SMA', 'MACD']


,date,open,high,low,close,volume,RSI,SMA,MACD
3199,2026-10-24,81167.328125,81175.664062,79277.710938,80870.507812,40821,44.260960,82613.214063,-166.226453
3200,2026-10-25,80868.015625,81018.898438,79857.710938,80121.289062,23751,41.644681,82330.728125,-311.760578
3201,2026-10-26,80126.765625,80799.187500,79397.078125,80119.875000,54048,41.639678,82059.103906,-422.343016
3202,2026-10-27,80129.570312,81250.109375,79496.843750,80891.789062,47442,45.489337,81937.586328,-442.591462
3203,2026-10-28,80829.859375,81773.453125,80405.039062,81331.335938,47700,47.608603,81917.465625,-418.348284


In [4]:
# Check required columns and missing values
required_columns = [
    "date", "close", "RSI", "SMA", "MACD"
]

missing_columns = [
    col for col in required_columns if col not in df.columns
]

print("Missing required columns:", missing_columns)
print("Missing values in required columns:")
print(df[required_columns].isnull().sum())

print("Duplicate dates:", df["date"].duplicated().sum())
print("Dates in ascending order:", df["date"].is_monotonic_increasing)


Missing required columns: []
Missing values in required columns:
date     0
close    0
RSI      0
SMA      0
MACD     0
dtype: int64
Duplicate dates: 0
Dates in ascending order: True


In [5]:
# Check that the saved model and scalers exist
required_files = [
    model_path,
    feature_scaler_path,
    target_scaler_path
]

for file_path in required_files:
    print(f"{file_path}: {'Exists' if file_path.exists() else 'NOT FOUND'}")

# Load the trained model and scalers
if all(file_path.exists() for file_path in required_files):
    final_model = load_model(model_path)
    feature_scaler_final = joblib.load(feature_scaler_path)
    target_scaler_final = joblib.load(target_scaler_path)

    print("\nModel and scalers loaded successfully.")
    print("Model input shape:", final_model.input_shape)
    print("Feature scaler expects features:", feature_scaler_final.n_features_in_)
    print("Target scaler expects features:", target_scaler_final.n_features_in_)
else:
    print("\nPlease check the model and scaler file paths.")

..\models\lstm_model.keras: Exists
..\models\feature_scaler.pkl: Exists
..\models\target_scaler.pkl: Exists



Model and scalers loaded successfully.
Model input shape: (None, 90, 8)
Feature scaler expects features: 8
Target scaler expects features: 1


In [6]:
# Select the latest 90 days of historical features
look_back = 90

feature_columns = [
    "open", "high", "low", "close",
    "volume", "RSI", "SMA", "MACD"
]

latest_data = df[feature_columns].tail(look_back).copy()
latest_dates = df["date"].tail(look_back).copy()

print("Input date range:", latest_dates.iloc[0].date(), "to", latest_dates.iloc[-1].date())
print("Input data shape:", latest_data.shape)
print("Expected shape: (90, 8)")

Input date range: 2026-07-31 to 2026-10-28
Input data shape: (90, 8)
Expected shape: (90, 8)


In [7]:
# Scale the latest 90 days using the scaler saved from 04
latest_data_scaled = feature_scaler_final.transform(latest_data)

# Reshape data for LSTM: (samples, timesteps, features)
latest_sequence = latest_data_scaled.reshape(
    1, look_back, len(feature_columns)
)

print("Scaled input shape:", latest_data_scaled.shape)
print("LSTM input shape:", latest_sequence.shape)

Scaled input shape: (90, 8)
LSTM input shape: (1, 90, 8)


In [8]:
# Check and initialize an empty prediction history file

prediction_dir = Path("../data/lstm/processed")
prediction_path = prediction_dir / "bitcoin_daily_predictions.csv"

prediction_columns = [
    "forecast_date",
    "prediction_created_at",
    "predicted_close",
    "actual_close",
    "prediction_error",
    "absolute_error"
]

if not prediction_path.exists() or prediction_path.stat().st_size == 0:
    prediction_history = pd.DataFrame(columns=prediction_columns)
    prediction_history.to_csv(prediction_path, index=False)
    print("Empty prediction history file initialized.")
else:
    prediction_history = pd.read_csv(prediction_path)
    print("Prediction history file contains data.")

print("Columns:", prediction_history.columns.tolist())
display(prediction_history.tail())

Prediction history file contains data.
Columns: ['forecast_date', 'prediction_created_at', 'predicted_close', 'actual_close', 'prediction_error', 'absolute_error']


,forecast_date,prediction_created_at,predicted_close,actual_close,prediction_error,absolute_error
0,2026-10-11,2026-10-11 15:17:23,82503.046875,83758.679688,-1255.632812,1255.632812
1,2026-10-12,2026-10-12 13:49:25,81637.734375,81842.289062,-204.554688,204.554688
2,2026-10-13,2026-10-13 14:15:19,80441.187500,83156.765625,-2715.578125,2715.578125
3,2026-10-27,2026-10-27 14:20:21,78575.109375,NaN,NaN,NaN


In [9]:
# Predict only if this forecast date has not been saved before

latest_date = latest_dates.iloc[-1]
forecast_date = latest_date + pd.Timedelta(days=1)

if prediction_path.exists():
    prediction_history = pd.read_csv(
        prediction_path,
        parse_dates=["forecast_date", "prediction_created_at"]
    )
    prediction_history["forecast_date"] = pd.to_datetime(
        prediction_history["forecast_date"],
        errors="coerce"
    ).dt.normalize()
    existing_record = prediction_history[
        prediction_history["forecast_date"] == forecast_date.normalize()
    ]

else:
    existing_record = pd.DataFrame()


if not existing_record.empty:
    # Reuse the original saved prediction
    saved_record = existing_record.iloc[0]

    next_close = float(saved_record["predicted_close"])

    # Get the latest historical close from the current dataset
    latest_date = latest_dates.iloc[-1]
    latest_close = float(df["close"].iloc[-1])

    print("Using existing saved prediction.")
    print("Latest historical date:", latest_date.date())
    print(f"Latest BTC Close Price: ${latest_close:,.2f}")
    print("Forecast date:", forecast_date.date())
    print(f"Predicted BTC Close Price: ${next_close:,.2f}")
    print(f"Predicted change: ${next_close - latest_close:,.2f}")
    print(
        f"Predicted change (%): "
        f"{(next_close / latest_close - 1) * 100:.2f}%"
    )

else:
    # Generate a new prediction
    next_close_scaled = final_model.predict(
        latest_sequence,
        verbose=0
    )

    next_close = target_scaler_final.inverse_transform(
        next_close_scaled
    )[0, 0]

    latest_date = latest_dates.iloc[-1]
    latest_close = df["close"].iloc[-1]

    print("New prediction generated.")
    print("Latest historical date:", latest_date.date())
    print(f"Latest BTC Close Price: ${latest_close:,.2f}")
    print("Forecast date:", forecast_date.date())
    print(f"Predicted BTC Close Price: ${next_close:,.2f}")
    print(f"Predicted change: {next_close - latest_close:,.2f}")
    print(
        f"Predicted change (%): "
        f"{(next_close / latest_close - 1) * 100:.2f}%"
    )

New prediction generated.
Latest historical date: 2026-10-28
Latest BTC Close Price: $81,331.34
Forecast date: 2026-10-29
Predicted BTC Close Price: $78,207.90
Predicted change: -3,123.44
Predicted change (%): -3.84%


In [10]:
# Define the daily prediction record file
prediction_dir = Path("../data/lstm/processed")
prediction_dir.mkdir(parents=True, exist_ok=True)

prediction_path = prediction_dir / "bitcoin_daily_predictions.csv"

# Create the current prediction record
new_prediction = pd.DataFrame([{
    "forecast_date": forecast_date,
    "prediction_created_at": pd.Timestamp.now().floor("s"),
    "predicted_close": next_close,
    "actual_close": np.nan,
    "prediction_error": np.nan,
    "absolute_error": np.nan
}])

# Load existing records, if available
if prediction_path.exists():
    prediction_history = pd.read_csv(
        prediction_path,
        parse_dates=["forecast_date", "prediction_created_at"]
    )
    prediction_history["forecast_date"] = pd.to_datetime(
        prediction_history["forecast_date"],
        errors="coerce"
    ).dt.normalize()

    # Preserve existing prediction for the same date
    if (
        prediction_history["forecast_date"]
        == forecast_date.normalize()
    ).any():
        print(f"Forecast for {forecast_date.date()} already exists.")
        print("Existing prediction record was preserved.")

    else:
        prediction_history = pd.concat(
            [prediction_history, new_prediction],
            ignore_index=True
        )
        prediction_history.to_csv(prediction_path, index=False)
        print("New prediction saved.")

else:
    prediction_history = new_prediction
    prediction_history.to_csv(prediction_path, index=False)
    print("Prediction history file created.")

print("Prediction file:", prediction_path)

New prediction saved.
Prediction file: ..\data\lstm\processed\bitcoin_daily_predictions.csv


### Update Actual Close and Prediction Error

Update the actual closing price only after the forecast date's daily candle is complete and the closing price is confirmed.

In [11]:
# Automatically update actual close and errors for past forecasts

# Load the original feature-engineered data
actual_df = pd.read_csv(data_path)
actual_df["date"] = pd.to_datetime(actual_df["date"]).dt.normalize()

# Load prediction history
prediction_history = pd.read_csv(
    prediction_path,
    parse_dates=[
        "forecast_date",
        "prediction_created_at"
    ]
)
prediction_history["forecast_date"] = (
    prediction_history["forecast_date"].dt.normalize()
)

today_date = pd.Timestamp.today().normalize()

# Find records that still need an actual close
pending_mask = (
    prediction_history["actual_close"].isna()
    & (prediction_history["forecast_date"] < today_date)
)

pending_records = prediction_history[pending_mask]

if pending_records.empty:
    print("No past predictions are waiting for actual close updates.")

else:
    updated_count = 0

    for idx, record in pending_records.iterrows():
        target_date = record["forecast_date"]

        # Only update if the forecast date exists in the data
        actual_row = actual_df[actual_df["date"] == target_date]

        if actual_row.empty:
            print(f"{target_date.date()}: actual close not available.")
            continue

        actual_close = float(actual_row.iloc[-1]["close"])
        predicted_close = float(record["predicted_close"])

        prediction_history.loc[idx, "actual_close"] = actual_close
        prediction_history.loc[idx, "prediction_error"] = (
            predicted_close - actual_close
        )
        prediction_history.loc[idx, "absolute_error"] = abs(
            predicted_close - actual_close
        )

        updated_count += 1

        print(
            f"{target_date.date()}: "
            f"Predicted=${predicted_close:,.2f}, "
            f"Actual=${actual_close:,.2f}, "
            f"Error=${predicted_close - actual_close:,.2f}"
        )

    # Save updated records
    prediction_history.to_csv(prediction_path, index=False)

    print(f"\nUpdated records: {updated_count}")
    print("Prediction history saved.")

2026-10-27: Predicted=$78,575.11, Actual=$80,891.79, Error=$-2,316.68

Updated records: 1
Prediction history saved.
